# Ship a Better Prompt Without Redeploying Code

Create a prompt, serve it in production, iterate to v2, evaluate it, and promote — via SDK, without touching application code.

By the end of this notebook you will have created a prompt, served it in production, iterated to v2, evaluated it against test cases, and promoted it live — without a single code redeploy.

**Prerequisites:**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` ([Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+

## Install

In [ ]:
%pip install futureagi ai-evaluation litellm --quiet

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"        # Replace with your key
os.environ["FI_SECRET_KEY"] = "your-secret-key"  # Replace with your key
os.environ["OPENAI_API_KEY"] = "your-openai-key"  # For litellm gpt-4o-mini calls

## Step 1: Create a prompt via SDK

In [ ]:
import os
from fi.prompt import Prompt
from fi.prompt.types import PromptTemplate, SystemMessage, UserMessage, ModelConfig

prompt_client = Prompt(
    template=PromptTemplate(
        name="support-response",
        messages=[
            SystemMessage(
                content="You are a helpful customer support agent for TechStore. "
                        "Answer the customer's question clearly and professionally."
            ),
            UserMessage(
                content="Customer question: {{question}}"
            ),
        ],
        model_configuration=ModelConfig(
            model_name="gpt-4o-mini",
            temperature=0.7,
            max_tokens=1000,
        ),
    ),
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

# Create the prompt as a draft and commit it as v1
prompt_client.create()
prompt_client.commit_current_version(
    message="Initial support prompt",
    label="production",
)

print(f"Created: {prompt_client.template.name} ({prompt_client.template.version})")

## Step 2: Serve the prompt in your application

`compile()` returns standard `[{"role": "system", "content": "..."}]` message dicts — compatible with any LLM provider via litellm.

In [ ]:
import os
import litellm
from fi.prompt import Prompt


def answer_question(question: str) -> str:
    prompt = Prompt.get_template_by_name(
        name="support-response",
        label="production",
        fi_api_key=os.environ["FI_API_KEY"],
        fi_secret_key=os.environ["FI_SECRET_KEY"],
    )

    messages = prompt.compile(question=question)

    response = litellm.completion(
        model="gpt-4o-mini",
        messages=messages,
    )
    return response.choices[0].message.content


print(answer_question("What is your return policy?"))

## Step 3: Create v2 with chain-of-thought reasoning

Each version can have its own model configuration. Here v2 uses a lower temperature for more deterministic chain-of-thought responses.

In [ ]:
from fi.prompt.types import PromptTemplate, SystemMessage, UserMessage, ModelConfig

prompt_client.create_new_version(
    template=PromptTemplate(
        name="support-response",
        messages=[
            SystemMessage(
                content="You are a precise customer support agent for TechStore.\n\n"
                        "Think through the customer's question step by step before answering:\n"
                        "1. What is the customer asking?\n"
                        "2. What information do I have that directly addresses this?\n"
                        "3. What is the clearest, most helpful response?"
            ),
            UserMessage(
                content="Customer question: {{question}}\n\nAnswer:"
            ),
        ],
        model_configuration=ModelConfig(
            model_name="gpt-4o-mini",
            temperature=0.3,
            max_tokens=1000,
        ),
    ),
    commit_message="Add chain-of-thought reasoning",
)

prompt_client.save_current_draft()
prompt_client.commit_current_version(message="v2: chain-of-thought prompt")

print(f"v2 created: {prompt_client.template.version}")

## Step 4: Test v2 before promoting it

We use `is_concise` here — for a support agent, concise answers are a key quality signal. You can swap in any of the 72+ [built-in eval metrics](https://docs.futureagi.com/future-agi/get-started/evaluation/builtin-evals/overview) like `groundedness`, `tone`, `completeness`, or `instruction_adherence` depending on what you want to measure.

In [ ]:
import litellm
from fi.evals import evaluate

test_cases = [
    "What is your return policy?",
    "How long does standard shipping take?",
    "Can I exchange a product instead of returning it?",
]

v2_prompt = Prompt.get_template_by_name(
    name="support-response",
    version="v2",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

print(f"{'Question':<45} {'Concise':>8}")
print("-" * 55)

for question in test_cases:
    messages = v2_prompt.compile(question=question)

    response = litellm.completion(
        model="gpt-4o-mini",
        messages=messages,
    )
    output = response.choices[0].message.content

    result = evaluate(
        "is_concise",
        output=output,
        model="turing_small",
    )
    print(f"{question[:43]:<45} {result.score:>8}")

## Step 5: Promote v2 to production

Your application now serves v2 on the next request — no redeploy. The `get_template_by_name(label="production")` call in Step 2 automatically picks up the new version.

In [ ]:
Prompt.assign_label_to_template_version(
    template_name="support-response",
    version="v2",
    label="production",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

print("v2 is now live in production.")

## Step 6: Rollback to v1

If v2 causes issues, reassign the production label back to v1. Your app picks up the change on the next request.

In [ ]:
Prompt.assign_label_to_template_version(
    template_name="support-response",
    version="v1",
    label="production",
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

print("Rolled back to v1.")

## Step 7: View version history

In [ ]:
versions = prompt_client.list_template_versions()

for v in versions:
    draft = "draft" if v.get("isDraft") else "committed"
    print(f"  {v['templateVersion']}  {draft}  {v['createdAt']}")

## What you built

- Created a `support-response` prompt via SDK and committed v1 with the production label
- Served the prompt in your app via `get_template_by_name(label="production")` + `compile()` with litellm
- Created v2 with chain-of-thought reasoning and a different `ModelConfig` (lower temperature)
- Evaluated v2 against test cases before promoting it
- Promoted v2 to production — your app picks it up on the next request, zero-downtime
- Rolled back to v1 by reassigning the production label
- Viewed version history with `list_template_versions()`

### Next steps

- [Experimentation](https://docs.futureagi.com/cookbook/quickstart/experimentation-compare-prompts) — A/B test prompts on a full dataset
- [Prompt Optimization](https://docs.futureagi.com/cookbook/quickstart/prompt-optimization) — automatically optimize your prompt using evaluation-driven search
- [Eval in CI/CD](https://docs.futureagi.com/cookbook/quickstart/cicd-eval-pipeline) — gate promotions on eval scores
- [Running Your First Eval](https://docs.futureagi.com/cookbook/quickstart/first-eval) — all evaluation metrics for scoring prompt variants